In [ ]:
# 공급업체 공급 처리 프로그램 interface

import random
import sqlite3
from datetime import datetime
from pathlib import Path


DEFAULT_DB_NAMES = ["파리바게트.db", "paris_baguette.db", "database.db"]


class SupplierCLI:
    def __init__(self, db_path):
        self.db_path = Path(db_path)
        self.conn = None
        self.supplier_code = None
        self.supplier_name = None

    # 1. DB 연결
    def connect(self):
        if not self.db_path.exists():
            raise FileNotFoundError(f"DB 파일을 찾을 수 없습니다: {self.db_path}")

        self.conn = sqlite3.connect(self.db_path)
        self.conn.row_factory = sqlite3.Row
        self.conn.execute("PRAGMA foreign_keys = ON;")
        self.validate_required_tables()

        print(f"\nDB 연결 완료: {self.db_path}")

    # 2. DB 연결 종료
    def close(self):
        if self.conn:
            self.conn.close()
            print("DB 연결을 종료했습니다.")

    # 3. 공급 처리에 필요한 테이블이 있는지 확인
    def validate_required_tables(self):
        required = {
            "공급업체",
            "브랜드",
            "발주",
            "발주상세",
            "공급내역",
            "보유",
            "상품",
            "지점",
        }

        rows = self.fetch_all(
            """
            SELECT name
            FROM sqlite_master
            WHERE type = 'table'
              AND name NOT LIKE 'sqlite_%';
            """
        )

        existing = {row["name"] for row in rows}
        missing = sorted(required - existing)

        if missing:
            print("공급 처리에 필요한 테이블이 일부 없습니다.")
            print("없는 테이블:", ", ".join(missing))

    # 4. SELECT 결과 여러 행 가져오기
    def fetch_all(self, sql, params=()):
        if self.conn is None:
            raise RuntimeError("DB 연결이 없습니다.")

        cur = self.conn.execute(sql, params)
        return cur.fetchall()

    # 5. SELECT 결과 한 행만 가져오기
    def fetch_one(self, sql, params=()):
        if self.conn is None:
            raise RuntimeError("DB 연결이 없습니다.")

        cur = self.conn.execute(sql, params)
        return cur.fetchone()

    # 6. 전체 메뉴 실행
    def run(self):
        self.connect()

        if not self.login():
            self.close()
            return

        while True:
            print_menu(self.supplier_name, self.supplier_code)
            choice = input("메뉴 번호를 입력하세요 > ").strip()

            try:
                if choice == "1":
                    self.show_my_info()
                elif choice == "2":
                    self.show_pending_orders()
                elif choice == "3":
                    self.show_order_detail()
                elif choice == "4":
                    self.complete_supply()
                elif choice == "5":
                    self.show_supply_history()
                elif choice == "6":
                    self.check_inventory_by_order()
                elif choice == "0":
                    print("프로그램을 종료합니다.")
                    break
                else:
                    print("목록에 있는 번호를 입력해주세요.")
            except sqlite3.Error as e:
                print(f"SQLite 오류가 발생했습니다: {e}")
            except Exception as e:
                print(f"오류가 발생했습니다: {e}")

        self.close()

    # 7. 공급업체 로그인
    def login(self):
        print("\n" + "=" * 60)
        print("공급업체 로그인")
        print("=" * 60)

        self.print_supplier_list()

        for _ in range(3):
            supplier_code = input("\n업체코드를 입력하세요. 종료하려면 0 입력 > ").strip()

            if supplier_code == "0":
                print("로그인을 취소했습니다.")
                return False

            row = self.fetch_one(
                """
                SELECT 업체코드, 업체명
                FROM 공급업체
                WHERE 업체코드 = ?;
                """,
                (supplier_code,),
            )

            if row:
                self.supplier_code = row["업체코드"]
                self.supplier_name = row["업체명"]
                print(f"\n로그인 완료: {self.supplier_name} ({self.supplier_code})")
                return True

            print("등록되지 않은 업체코드입니다. 다시 입력해주세요.")

        print("로그인에 여러 번 실패해서 프로그램을 종료합니다.")
        return False

    # 8. 로그인할 때 참고할 공급업체 목록 출력
    def print_supplier_list(self):
        rows = self.fetch_all(
            """
            SELECT 업체코드, 업체명, 담당자명, 연락처
            FROM 공급업체
            ORDER BY 업체코드;
            """
        )

        if not rows:
            print("등록된 공급업체가 없습니다.")
            return

        print("\n[공급업체 목록 참고]")
        print_rows(rows)

    # 9. 내 업체 정보 조회
    def show_my_info(self):
        print("\n[1] 내 업체 정보 조회")

        row = self.fetch_one(
            """
            SELECT 업체코드, 업체명, 담당자명, 연락처
            FROM 공급업체
            WHERE 업체코드 = ?;
            """,
            (self.supplier_code,),
        )

        print_rows([row] if row else [])

        brand_rows = self.fetch_all(
            """
            SELECT b.브랜드코드, b.브랜드명, b.제조사정보
            FROM 취급 c
            JOIN 브랜드 b ON c.브랜드코드 = b.브랜드코드
            WHERE c.업체코드 = ?
            ORDER BY b.브랜드명;
            """,
            (self.supplier_code,),
        )

        print("\n[취급 브랜드]")
        print_rows(brand_rows)

        product_rows = self.fetch_all(
            """
            SELECT p.상품코드, p.이름 AS 상품명, b.브랜드명, p.가격
            FROM 공급 s
            JOIN 상품 p ON s.상품코드 = p.상품코드
            LEFT JOIN 브랜드 b ON p.브랜드코드 = b.브랜드코드
            WHERE s.업체코드 = ?
            ORDER BY b.브랜드명, p.이름
            LIMIT 30;
            """,
            (self.supplier_code,),
        )

        print("\n[공급 가능 상품 일부]")
        print_rows(product_rows)

    # 10. 미처리 발주 목록 조회
    def show_pending_orders(self):
        print("\n[2] 미처리 발주 목록 조회")

        rows = self.fetch_all(
            """
            SELECT
                o.발주번호,
                o.발주일자,
                o.상태,
                o.지점명,
                j.지역_시도,
                COUNT(od.항목번호) AS 발주항목수,
                SUM(od.주문수량) AS 총주문수량
            FROM 발주 o
            JOIN 지점 j ON o.지점명 = j.지점명
            LEFT JOIN 발주상세 od ON o.발주번호 = od.발주번호
            WHERE o.업체코드 = ?
              AND o.상태 <> '완료'
            GROUP BY o.발주번호, o.발주일자, o.상태, o.지점명, j.지역_시도
            ORDER BY o.발주일자, o.발주번호;
            """,
            (self.supplier_code,),
        )

        if not rows:
            print("현재 처리할 발주가 없습니다.")
            return []

        print_rows(rows)
        return rows

    # 11. 발주 상세 조회
    def show_order_detail(self):
        print("\n[3] 발주 상세 조회")
        print("먼저 현재 로그인한 업체의 미처리 발주 목록을 보여줍니다.")
        print("아래 목록에 있는 발주번호를 그대로 입력하면 상세 내용을 볼 수 있습니다.")

        pending_orders = self.show_pending_orders()

        if not pending_orders:
            return

        order_no = input("\n조회할 발주번호를 입력하세요 > ").strip()
        order_no = self.normalize_order_no(order_no)

        if not order_no:
            print("발주번호를 입력해야 합니다.")
            return

        if not self.check_order_access(order_no):
            print("해당 발주번호가 없거나, 현재 로그인한 업체의 발주가 아닙니다.")
            print("위 목록에 있는 발주번호를 그대로 입력했는지 확인해주세요.")
            return

        order = self.fetch_one(
            """
            SELECT 상태
            FROM 발주
            WHERE 발주번호 = ?
              AND 업체코드 = ?;
            """,
            (order_no, self.supplier_code),
        )

        if order and order["상태"] == "완료":
            print("이미 완료 처리된 발주입니다. 공급 이력 조회 메뉴에서 확인하는 것이 더 적절합니다.")
            return

        self.print_order_header(order_no)
        self.print_order_items(order_no)

    # 12. 공급 완료 처리
    def complete_supply(self):
        print("\n[4] 공급 완료 처리")
        print("공급 완료 처리할 수 있는 미처리 발주 목록을 먼저 보여줍니다.")
        pending_orders = self.show_pending_orders()

        if not pending_orders:
            return

        order_no = input("\n완료 처리할 발주번호를 입력하세요. 취소는 Enter > ").strip()
        order_no = self.normalize_order_no(order_no)

        if not order_no:
            print("공급 완료 처리를 취소했습니다.")
            return

        order = self.fetch_one(
            """
            SELECT 발주번호, 발주일자, 상태, 지점명, 업체코드
            FROM 발주
            WHERE 발주번호 = ?
              AND 업체코드 = ?;
            """,
            (order_no, self.supplier_code),
        )

        if not order:
            print("해당 발주번호가 없거나, 현재 로그인한 업체의 발주가 아닙니다.")
            return

        if order["상태"] == "완료":
            print("이미 완료 처리된 발주입니다.")
            return

        items = self.fetch_order_items_for_supply(order_no, order["지점명"])

        if not items:
            print("발주상세 항목이 없어 공급 완료 처리를 할 수 없습니다.")
            return

        print("\n[처리할 발주 정보]")
        self.print_order_header(order_no)

        print("\n[공급 처리할 상품]")
        print_rows(items)

        default_qty = sum(row["주문수량"] for row in items)
        supply_qty = input_int(f"공급수량을 입력하세요. Enter={default_qty}", default_qty)
        supplied_at = input_datetime("공급일시를 입력하세요. Enter=현재시각")

        if supply_qty != default_qty:
            print("\n입력한 공급수량이 발주상세 주문수량 합계와 다릅니다.")
            print("공급내역에는 입력한 수량이 저장되고, 재고는 상품별 주문수량 기준으로 반영됩니다.")
            confirm_qty = input("이대로 진행할까요? (y/N) > ").strip().lower()

            if confirm_qty != "y":
                print("공급 완료 처리를 취소했습니다.")
                return

        print("\n공급 완료 처리를 하면 다음 작업이 진행됩니다.")
        print("- 공급내역에 새 공급 기록 추가")
        print("- 발주 상태를 '대기'에서 '완료'로 변경")
        print("- 발주상세의 주문수량만큼 매장 재고 증가")

        confirm = input("정말 공급 완료 처리할까요? (y/N) > ").strip().lower()

        if confirm != "y":
            print("공급 완료 처리를 취소했습니다.")
            return

        supply_no = self.generate_supply_no()

        try:
            self.conn.execute("BEGIN IMMEDIATE;")

            # 공급내역 INSERT
            self.conn.execute(
                """
                INSERT INTO 공급내역 (공급번호, 공급일시, 공급수량, 발주번호, 업체코드)
                VALUES (?, ?, ?, ?, ?);
                """,
                (supply_no, supplied_at, supply_qty, order_no, self.supplier_code),
            )

            # 발주상세에 있는 상품 수량만큼 보유 재고 증가
            for item in items:
                current_inventory = item["현재재고수량"]

                if current_inventory is None:
                    self.conn.execute(
                        """
                        INSERT INTO 보유 (지점명, 상품코드, 재고수량, 최소재고기준, 매장가격)
                        SELECT ?, p.상품코드, ?, 10, p.가격
                        FROM 상품 p
                        WHERE p.상품코드 = ?;
                        """,
                        (order["지점명"], item["주문수량"], item["상품코드"]),
                    )
                else:
                    self.conn.execute(
                        """
                        UPDATE 보유
                        SET 재고수량 = 재고수량 + ?
                        WHERE 지점명 = ?
                          AND 상품코드 = ?;
                        """,
                        (item["주문수량"], order["지점명"], item["상품코드"]),
                    )

            # 발주 상태를 완료로 변경
            self.conn.execute(
                """
                UPDATE 발주
                SET 상태 = '완료'
                WHERE 발주번호 = ?
                  AND 업체코드 = ?;
                """,
                (order_no, self.supplier_code),
            )

            self.conn.commit()

        except Exception:
            self.conn.rollback()
            raise

        print("\n공급 완료 처리가 끝났습니다.")
        print(f"공급번호: {supply_no}")
        print(f"공급일시: {supplied_at}")
        print(f"공급수량: {supply_qty}")
        print("발주 상태: 완료")
        print("재고 반영: 완료")

        print("\n[재고 반영 결과]")
        self.print_inventory_by_order(order_no)

    # 13. 공급 이력 조회
    def show_supply_history(self):
        print("\n[5] 공급 이력 조회")
        print("아무 조건도 입력하지 않고 Enter를 누르면 현재 업체의 전체 공급 이력이 조회됩니다.")
        print("특정 발주만 보고 싶으면 발주번호를 입력하면 됩니다.")

        order_keyword = input("발주번호 검색어, 전체는 Enter > ").strip()

        # 00082처럼 숫자만 입력해도 O0082 형태까지 같이 찾도록 처리
        if order_keyword.isdigit():
            order_keyword = "O" + order_keyword[-4:].zfill(4)
            print(f"발주번호를 {order_keyword} 기준으로 조회합니다.")

        start_date = input("시작일 YYYY-MM-DD, 전체는 Enter > ").strip()
        end_date = input("종료일 YYYY-MM-DD, 전체는 Enter > ").strip()
        store_keyword = input("지점명 검색어, 전체는 Enter > ").strip()
        brand_keyword = input("브랜드명 검색어, 전체는 Enter > ").strip()

        conditions = ["s.업체코드 = ?"]
        params = [self.supplier_code]

        if order_keyword:
            conditions.append("s.발주번호 LIKE ?")
            params.append(f"%{order_keyword}%")

        if start_date:
            conditions.append("date(s.공급일시) >= ?")
            params.append(start_date)

        if end_date:
            conditions.append("date(s.공급일시) <= ?")
            params.append(end_date)

        if store_keyword:
            conditions.append("o.지점명 LIKE ?")
            params.append(f"%{store_keyword}%")

        if brand_keyword:
            conditions.append("b.브랜드명 LIKE ?")
            params.append(f"%{brand_keyword}%")

        where_sql = "WHERE " + " AND ".join(conditions)

        rows = self.fetch_all(
            f"""
            SELECT
                s.공급번호,
                s.공급일시,
                s.공급수량,
                s.발주번호,
                o.발주일자,
                o.상태 AS 발주상태,
                o.지점명,
                b.브랜드명,
                p.상품코드,
                p.이름 AS 상품명,
                od.주문수량
            FROM 공급내역 s
            JOIN 발주 o ON s.발주번호 = o.발주번호
            JOIN 발주상세 od ON o.발주번호 = od.발주번호
            JOIN 상품 p ON od.상품코드 = p.상품코드
            LEFT JOIN 브랜드 b ON p.브랜드코드 = b.브랜드코드
            {where_sql}
            ORDER BY s.공급일시 DESC, s.공급번호 DESC, od.항목번호;
            """,
            params,
        )

        print_rows(rows)

    # 14. 재고 반영 확인
    def check_inventory_by_order(self):
        print("\n[6] 재고 반영 확인")
        order_no = input("확인할 발주번호를 입력하세요 > ").strip()
        order_no = self.normalize_order_no(order_no)

        if not order_no:
            print("발주번호를 입력해야 합니다.")
            return

        self.print_inventory_by_order(order_no)

    # 15. 발주번호 기준으로 재고 반영 결과 출력
    def print_inventory_by_order(self, order_no):
        order = self.fetch_one(
            """
            SELECT 발주번호, 지점명, 상태, 업체코드
            FROM 발주
            WHERE 발주번호 = ?
              AND 업체코드 = ?;
            """,
            (order_no, self.supplier_code),
        )

        if not order:
            print("해당 발주번호가 없거나, 현재 로그인한 업체의 발주가 아닙니다.")
            print("미처리 발주 목록이나 공급 이력 조회에 나온 발주번호를 그대로 입력했는지 확인해주세요.")
            return

        rows = self.fetch_all(
            """
            SELECT
                od.발주번호,
                o.지점명,
                od.상품코드,
                p.이름 AS 상품명,
                od.주문수량,
                h.재고수량 AS 현재재고수량,
                h.최소재고기준,
                CASE
                    WHEN h.재고수량 IS NULL THEN '보유정보 없음'
                    WHEN h.재고수량 < h.최소재고기준 THEN '재고부족'
                    ELSE '정상'
                END AS 재고상태
            FROM 발주상세 od
            JOIN 발주 o ON od.발주번호 = o.발주번호
            JOIN 상품 p ON od.상품코드 = p.상품코드
            LEFT JOIN 보유 h ON h.지점명 = o.지점명 AND h.상품코드 = od.상품코드
            WHERE od.발주번호 = ?
              AND o.업체코드 = ?
            ORDER BY od.항목번호;
            """,
            (order_no, self.supplier_code),
        )

        print(f"\n발주번호: {order_no}")
        print(f"지점명: {order['지점명']}")
        print(f"발주상태: {order['상태']}")
        print_rows(rows)

    # 16. 공급 처리용 발주상세 조회
    def fetch_order_items_for_supply(self, order_no, store_name):
        return self.fetch_all(
            """
            SELECT
                od.발주번호,
                od.항목번호,
                od.상품코드,
                p.이름 AS 상품명,
                b.브랜드명,
                od.주문수량,
                h.재고수량 AS 현재재고수량,
                h.최소재고기준
            FROM 발주상세 od
            JOIN 상품 p ON od.상품코드 = p.상품코드
            LEFT JOIN 브랜드 b ON p.브랜드코드 = b.브랜드코드
            LEFT JOIN 보유 h ON h.지점명 = ? AND h.상품코드 = od.상품코드
            WHERE od.발주번호 = ?
            ORDER BY od.항목번호;
            """,
            (store_name, order_no),
        )

    # 17. 발주 기본 정보 출력
    def print_order_header(self, order_no):
        rows = self.fetch_all(
            """
            SELECT
                o.발주번호,
                o.발주일자,
                o.상태,
                o.지점명,
                j.지역_시도,
                j.주소,
                s.업체명
            FROM 발주 o
            JOIN 지점 j ON o.지점명 = j.지점명
            JOIN 공급업체 s ON o.업체코드 = s.업체코드
            WHERE o.발주번호 = ?
              AND o.업체코드 = ?;
            """,
            (order_no, self.supplier_code),
        )

        print_rows(rows)

    # 18. 발주 상품 목록 출력
    def print_order_items(self, order_no):
        rows = self.fetch_all(
            """
            SELECT
                od.항목번호,
                od.상품코드,
                p.이름 AS 상품명,
                b.브랜드명,
                od.주문수량,
                h.재고수량 AS 현재재고수량,
                h.최소재고기준
            FROM 발주상세 od
            JOIN 상품 p ON od.상품코드 = p.상품코드
            LEFT JOIN 브랜드 b ON p.브랜드코드 = b.브랜드코드
            JOIN 발주 o ON od.발주번호 = o.발주번호
            LEFT JOIN 보유 h ON h.지점명 = o.지점명 AND h.상품코드 = od.상품코드
            WHERE od.발주번호 = ?
            ORDER BY od.항목번호;
            """,
            (order_no,),
        )

        print("\n[발주상세]")
        print_rows(rows)

    # 19. 발주번호 입력값 정리
    def normalize_order_no(self, order_no):
        order_no = order_no.strip()

        if not order_no:
            return order_no

        # DB에는 O0082처럼 앞에 O가 붙어 있으므로,
        # 사용자가 00082처럼 숫자만 입력하면 O0082로 바꿔서 확인
        candidates = [order_no]

        if order_no.isdigit():
            candidates.append("O" + order_no[-4:].zfill(4))

        if order_no.upper().startswith("O"):
            candidates.append("O" + order_no[1:].zfill(4))

        for candidate in candidates:
            row = self.fetch_one(
                """
                SELECT 발주번호
                FROM 발주
                WHERE 발주번호 = ?
                  AND 업체코드 = ?;
                """,
                (candidate, self.supplier_code),
            )

            if row:
                if candidate != order_no:
                    print(f"입력한 발주번호를 {candidate}로 확인합니다.")
                return candidate

        return order_no

    # 20. 로그인한 업체의 발주인지 확인
    def check_order_access(self, order_no):
        row = self.fetch_one(
            """
            SELECT 발주번호
            FROM 발주
            WHERE 발주번호 = ?
              AND 업체코드 = ?;
            """,
            (order_no, self.supplier_code),
        )

        return row is not None

    # 21. 공급번호 만들기
    def generate_supply_no(self):
        while True:
            supply_no = "SI" + datetime.now().strftime("%y%m%d%H%M%S") + str(random.randint(10, 99))

            exists = self.fetch_one(
                """
                SELECT 공급번호
                FROM 공급내역
                WHERE 공급번호 = ?;
                """,
                (supply_no,),
            )

            if not exists:
                return supply_no


# 메뉴 출력
def print_menu(supplier_name, supplier_code):
    print("\n" + "=" * 70)
    print(f"공급업체 공급 처리 - {supplier_name} ({supplier_code})")
    print("=" * 70)
    print("1. 내 업체 정보 조회")
    print("2. 미처리 발주 목록 조회")
    print("3. 발주 상세 조회")
    print("4. 공급 완료 처리")
    print("5. 공급 이력 조회")
    print("6. 재고 반영 확인")
    print("0. 종료")
    print("=" * 70)


# 조회 결과를 표처럼 출력
def print_rows(rows, max_width=24):
    if not rows:
        print("조회 결과가 없습니다.")
        return

    headers = list(rows[0].keys())
    str_rows = []

    for row in rows:
        str_rows.append([format_cell(row[h], max_width) for h in headers])

    widths = []

    for idx, header in enumerate(headers):
        width = max(len(header), *(len(row[idx]) for row in str_rows))
        widths.append(min(width, max_width))

    header_line = " | ".join(headers[i].ljust(widths[i]) for i in range(len(headers)))
    print("\n" + header_line)
    print("-" * len(header_line))

    for row in str_rows:
        print(" | ".join(row[i].ljust(widths[i]) for i in range(len(headers))))

    print(f"\n총 {len(rows)}행")


# 너무 긴 값은 짧게 줄여서 보여주기
def format_cell(value, max_width):
    if value is None:
        text = "NULL"
    else:
        text = str(value)

    text = text.replace("\n", " ")

    if len(text) > max_width:
        return text[: max_width - 3] + "..."

    return text


# 숫자 입력 받기
def input_int(prompt, default):
    while True:
        raw = input(f"{prompt} > ").strip()

        if not raw:
            return default

        try:
            value = int(raw)
        except ValueError:
            print("숫자로 입력해주세요.")
            continue

        if value <= 0:
            print("1 이상의 숫자를 입력해주세요.")
            continue

        return value


# 공급일시 입력 받기
def input_datetime(prompt):
    raw = input(f"{prompt} > ").strip()

    if not raw:
        return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    for fmt in ("%Y-%m-%d %H:%M:%S", "%Y-%m-%d"):
        try:
            parsed = datetime.strptime(raw, fmt)

            if fmt == "%Y-%m-%d":
                return parsed.strftime("%Y-%m-%d 00:00:00")

            return parsed.strftime("%Y-%m-%d %H:%M:%S")
        except ValueError:
            pass

    print("날짜 형식이 맞지 않아 현재 시각으로 처리합니다.")
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


# DB 파일 자동으로 찾기
def discover_db_paths():
    paths = []

    cwd = Path.cwd()
    home = Path.home()

    # 프로젝트 폴더 위치가 바뀌어도 찾을 수 있도록 여러 위치를 후보로 둠
    project_roots = [
        cwd,
        cwd.parent,
        home / "paris-baguette-system-db",
        home / "Desktop" / "paris-baguette-system-db",
    ]

    # 같은 위치가 여러 번 들어가지 않게 정리
    unique_roots = []

    for root in project_roots:
        if root not in unique_roots:
            unique_roots.append(root)

    for root in unique_roots:
        search_dirs = [
            root / "schema",
            root / "data",
            root,
        ]

        for base in search_dirs:
            for name in DEFAULT_DB_NAMES:
                candidate = base / name

                if candidate.exists() and candidate not in paths:
                    paths.append(candidate)

    return paths


# 사용할 DB 파일 선택
def select_db_path():
    found = discover_db_paths()

    if found:
        # schema 폴더 안의 파리바게트.db를 가장 먼저 사용
        for path in found:
            if path.name == "파리바게트.db" and path.parent.name == "schema":
                print(f"DB 파일 자동 선택: {path}")
                return str(path)

        print(f"DB 파일 자동 선택: {found[0]}")
        return str(found[0])

    print("DB 파일을 자동으로 찾지 못했습니다.")
    print("예: schema/파리바게트.db")
    return input("사용할 DB 파일 경로를 입력하세요 > ").strip()


# 프로그램 시작
def main():
    print("공급업체 공급 처리 프로그램을 시작합니다.")

    db_path = select_db_path()

    if not db_path:
        print("DB 파일을 선택하지 않아 종료합니다.")
        return

    cli = SupplierCLI(db_path)
    cli.run()


if __name__ == "__main__":
    main()


공급업체 공급 처리 프로그램을 시작합니다.
DB 파일 자동 선택: /Users/kimmiso/paris-baguette-system-db/schema/파리바게트.db

DB 연결 완료: /Users/kimmiso/paris-baguette-system-db/schema/파리바게트.db

공급업체 로그인

[공급업체 목록 참고]

업체코드  | 업체명      | 담당자명 | 연락처         
--------------------------------------
SP001 | SPC GFS  | 김민준  | 02-1234-5678
SP002 | 서울우유협동조합 | 이서연  | 02-2345-6789
SP003 | 동서식품     | 박지호  | 02-3456-7890
SP004 | 롯데칠성음료   | 최수아  | 02-4567-8901
SP005 | 오리온      | 정예준  | 02-5678-9012

총 5행



업체코드를 입력하세요. 종료하려면 0 입력 >  SP001



로그인 완료: SPC GFS (SP001)

공급업체 공급 처리 - SPC GFS (SP001)
1. 내 업체 정보 조회
2. 미처리 발주 목록 조회
3. 발주 상세 조회
4. 공급 완료 처리
5. 공급 이력 조회
6. 재고 반영 확인
0. 종료


메뉴 번호를 입력하세요 >  1



[1] 내 업체 정보 조회

업체코드  | 업체명     | 담당자명 | 연락처         
-------------------------------------
SP001 | SPC GFS | 김민준  | 02-1234-5678

총 1행

[취급 브랜드]

브랜드코드 | 브랜드명  | 제조사정보
---------------------
BR001 | 파리바게트 | SPC그룹
BR002 | 파리크라상 | SPC그룹

총 2행

[공급 가능 상품 일부]

상품코드 | 상품명     | 브랜드명  | 가격  
-----------------------------
P023 | 계피롤     | 파리바게트 | 3334
P013 | 고구마빵    | 파리바게트 | 2639
P022 | 꽈배기     | 파리바게트 | 2053
P003 | 단팥빵     | 파리바게트 | 3425
P021 | 도넛      | 파리바게트 | 2857
P008 | 마늘바게트   | 파리바게트 | 2481
P019 | 머핀      | 파리바게트 | 2470
P006 | 멜론빵     | 파리바게트 | 3883
P005 | 모카빵     | 파리바게트 | 1533
P028 | 밤빵      | 파리바게트 | 2122
P018 | 버터롤     | 파리바게트 | 3481
P010 | 베이글     | 파리바게트 | 2757
P024 | 블루베리머핀  | 파리바게트 | 3587
P002 | 소금빵     | 파리바게트 | 2887
P016 | 슈크림빵    | 파리바게트 | 2725
P020 | 스콘      | 파리바게트 | 2409
P007 | 식빵      | 파리바게트 | 2864
P027 | 연유빵     | 파리바게트 | 2773
P012 | 옥수수빵    | 파리바게트 | 3486
P025 | 초코크루아상  | 파리바게트 | 1576
P009 | 치아바타    | 파리바게트 | 3625
P017 | 치즈빵     | 파리바게트 | 2731
P014 | 카레빵     | 파리바게트

...


[4] 공급 완료 처리
공급 완료 처리할 수 있는 미처리 발주 목록을 먼저 보여줍니다.

[2] 미처리 발주 목록 조회

발주번호    | 발주일자       | 상태 | 지점명    | 지역_시도 | 발주항목수 | 총주문수량
----------------------------------------------------------
O0026   | 2026-03-10 | 대기 | 판교점    | 경기    | 2     | 110  
O0015   | 2026-03-16 | 대기 | 광주점    | 광주    | 4     | 200  
O0009   | 2026-03-22 | 대기 | 이태원점   | 서울    | 3     | 141  
O0069   | 2026-04-03 | 대기 | 광주점    | 광주    | 1     | 70   
O0021   | 2026-04-26 | 대기 | 판교점    | 경기    | 5     | 225  
O0052   | 2026-04-27 | 대기 | 대구점    | 대구    | 4     | 242  
O0044   | 2026-04-28 | 대기 | 인천점    | 인천    | 5     | 275  
O0089   | 2026-04-29 | 대기 | 홍대점    | 서울    | 1     | 31   
O0094   | 2026-05-07 | 대기 | 부산해운대점 | 부산    | 3     | 173  
O0013   | 2026-05-09 | 대기 | 대전점    | 대전    | 3     | 178  
O0006   | 2026-05-20 | 대기 | 홍대점    | 서울    | 5     | 230  
O0056   | 2026-05-20 | 대기 | 신촌점    | 서울    | 4     | 152  
O0025   | 2026-06-03 | 대기 | 잠실점    | 서울    | 2     | 152  
O0070   | 2026-06-14 | 대기 | 판교점    | 경기    | 1


완료 처리할 발주번호를 입력하세요. 취소는 Enter >  00026


입력한 발주번호를 O0026로 확인합니다.

[처리할 발주 정보]

발주번호  | 발주일자       | 상태 | 지점명 | 지역_시도 | 주소                | 업체명    
-------------------------------------------------------------------
O0026 | 2026-03-10 | 대기 | 판교점 | 경기    | 경기 성남시 분당구 판교로 88 | SPC GFS

총 1행

[공급 처리할 상품]

발주번호  | 항목번호 | 상품코드 | 상품명     | 브랜드명  | 주문수량 | 현재재고수량 | 최소재고기준
--------------------------------------------------------------
O0026 | 1    | P026 | 햄치즈크루아상 | 파리바게트 | 76   | 84     | 6     
O0026 | 2    | P010 | 베이글     | 파리바게트 | 34   | 100    | 6     

총 2행


공급수량을 입력하세요. Enter=110 >  
공급일시를 입력하세요. Enter=현재시각 >  



공급 완료 처리를 하면 다음 작업이 진행됩니다.
- 공급내역에 새 공급 기록 추가
- 발주 상태를 '대기'에서 '완료'로 변경
- 발주상세의 주문수량만큼 매장 재고 증가
